# Abstention drift: Colab quickstart
Runtime → Change runtime type → **T4 GPU**. Runs the whole pipeline on one domain (FiQA) so the team can iterate fast.

In [ ]:
!git clone <YOUR-REPO-URL> abstention-drift
%cd abstention-drift
!pip -q install -r requirements.txt && pip -q install -e .
!bash scripts/setup_data.sh

In [ ]:
!python scripts/dataset_stats.py
!pytest -q

In [ ]:
# indexes for one domain (BM25 + BGE dense), ~5 min on T4
!python scripts/build_indexes.py --collections fiqa

In [ ]:
# checker data: human rewrites (fast); switch to --rewrite llm for the real runs
!python scripts/make_checker_data.py --rewrite oracle --collections fiqa

In [ ]:
!python scripts/train_checker.py --set checker.epochs=2

In [ ]:
# small LLM for quick iteration; use the 7B default for reported numbers
!python scripts/run_pipeline.py --system S4 --split dev --limit 60 --collections fiqa --set llm.model=Qwen/Qwen2.5-1.5B-Instruct
!python scripts/run_pipeline.py --system S2 --split dev --limit 60 --collections fiqa --set llm.model=Qwen/Qwen2.5-1.5B-Instruct

In [ ]:
!python scripts/evaluate_abstention.py outputs/S2_dev.jsonl outputs/S4_dev.jsonl
from IPython.display import Image; Image('results/drift.png')